# [실습 01] 오픈웨이트 LLM으로 '두뇌' 체험하기

> **연계**: 제1부 01장 · **환경**: Google Colab · **모델**: 오픈웨이트 `Qwen/Qwen2.5-0.5B-Instruct` (Hugging Face)

**학습 목표**
- 오픈웨이트 LLM을 Hugging Face `transformers`로 직접 불러와 실행한다.
- LLM이 에이전트의 '두뇌(추론 엔진)'라는 것을 체험한다.
- '그냥 답하기' vs '지시 형식을 따르기(지시 따르기 역량)'의 차이를 관찰한다.

## 0. 환경 준비

Colab 상단 메뉴 `런타임 > 런타임 유형 변경`에서 **T4 GPU**를 선택하면 더 빠릅니다. (CPU로도 실행됩니다.)

In [ ]:
!pip install -q transformers accelerate torch

## 1. 오픈웨이트 모델 불러오기 (LLM = 두뇌)

가중치가 공개된 **오픈웨이트** 모델을 내려받아 파이프라인으로 로드합니다. (07-1 절 참고)

In [ ]:
import torch
from transformers import pipeline

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # 오픈웨이트, 소형 → Colab에서 실행 가능
generator = pipeline(
    "text-generation",
    model=MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print("모델 로드 완료:", MODEL)

## 2. 첫 응답 생성하기

LLM은 '언어 이해·생성'이라는 기본 역량을 가진 두뇌입니다. (01-2 절)

In [ ]:
messages = [
    {"role": "system", "content": "당신은 친절한 한국어 도우미입니다."},
    {"role": "user", "content": "AI 에이전트가 무엇인지 한 문장으로 쉽게 설명해줘."},
]
out = generator(messages, max_new_tokens=120, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

## 3. '지시 따르기' 역량 체험 (에이전트의 핵심)

에이전트는 도구를 정확한 **형식**으로 호출해야 합니다. 그 토대가 '지시 따르기' 역량입니다. (01-2 절)
아래처럼 **출력 형식을 지정**하면 모델이 형식을 지켜 답하는 것을 관찰하세요.

In [ ]:
prompt = '''다음 문장의 감성을 분석해 반드시 JSON만 출력하라.
형식: {"sentiment": "긍정|부정|중립", "reason": "..."}
문장: "이 영화 정말 재미있고 감동적이었어요!"'''

messages = [{"role": "user", "content": prompt}]
out = generator(messages, max_new_tokens=120, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

## 4. 정리

- 오픈웨이트 LLM을 **직접 내려받아** Colab에서 실행했다. (도구: Hugging Face `transformers`)
- LLM은 언어를 **이해·생성**하는 두뇌이며, **지시 따르기**로 형식을 지킬 수 있다.
- 이 '지시 따르기'가 3장(도구 사용)·4장(ReAct)에서 도구를 정확히 호출하는 토대가 된다.

**더 해보기**: `MODEL`을 `Qwen/Qwen2.5-1.5B-Instruct`로 바꿔 응답 품질 차이를 비교해 보세요.